# Panduan Praktikum Data Mining: EDA, Visualisasi, dan Modeling

**Tujuan:** memahami bukan hanya kode yang dijalankan, tetapi juga *apa gunanya, mengapa diperlukan, dan bagaimana logikanya*.

Dataset yang digunakan adalah `Telco_Churn_Latihan.csv`, dataset sintetis untuk latihan. Target kita adalah mempelajari pola pelanggan yang `Churn = Yes` atau `No`.

## Cara menggunakan notebook
1. Letakkan file CSV di folder yang sama dengan notebook.
2. Jalankan cell dari atas ke bawah.
3. Baca markdown sebelum menjalankan kode.
4. Setelah grafik muncul, coba jelaskan dengan kata-katamu sendiri apa yang terlihat.
5. Ingat: grafik membantu kita bertanya dan menemukan pola; grafik tidak otomatis membuktikan sebab-akibat.

**Alur belajar:** memuat data → memeriksa struktur → visualisasi keseluruhan → membersihkan data → membagi train/test → preprocessing → modeling → evaluasi → cross-validation → imbalance → tuning.

**Edisi tambahan:** lab preprocessing data mentah, format angka Indonesia, feature engineering, dan integrasi pipeline.

## 0. Import library

Sebelum membuat grafik atau melatih model, kita perlu mengimpor alat yang dibutuhkan.

- `pandas` (`pd`) dipakai untuk membaca dan mengolah data berbentuk tabel.
- `numpy` (`np`) membantu operasi numerik.
- `matplotlib.pyplot` (`plt`) menyediakan fungsi dasar untuk membuat grafik.
- `seaborn` (`sns`) mempermudah pembuatan grafik statistik yang lebih informatif.

**Logika:** kita mengimpor library satu kali di awal agar fungsi-fungsinya bisa digunakan pada cell berikutnya.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set_theme(style="whitegrid")

## 1. Memuat CSV dan melihat bentuk dataset

`pd.read_csv(...)` membaca file CSV dan menyimpannya sebagai DataFrame bernama `df`.

- `df.head()` menampilkan 5 baris pertama agar kita bisa mengintip isi data.
- `df.shape` mengembalikan `(jumlah_baris, jumlah_kolom)`.
- `df.info()` menampilkan nama kolom, jumlah nilai non-kosong, dan tipe data.
- `df.sample(5, random_state=42)` mengambil contoh 5 baris secara acak. `random_state` membuat contoh acaknya konsisten.

**Kenapa ini penting?** Kita belum seharusnya langsung melatih model sebelum tahu struktur dan tipe datanya. Sesuaikan nama file dengan lokasi CSV milikmu.

In [ ]:
df = pd.read_csv("Telco_Churn_Latihan.csv")

display(df.head())
print("Jumlah baris dan kolom:", df.shape)
df.info()
display(df.sample(5, random_state=42))

## 2. Ringkasan kualitas data

Bagian ini memeriksa masalah yang mungkin perlu ditangani sebelum modeling.

- `isna().sum()` menghitung nilai kosong (`NaN`) di setiap kolom.
- `duplicated().sum()` menghitung baris yang benar-benar duplikat.
- `nunique()` menghitung banyaknya nilai unik pada setiap kolom.
- `describe(include="all")` memberi ringkasan statistik; hasil untuk numerik dan kategorikal tidak sama.

**Catatan:** string kosong seperti `""` tidak selalu dianggap sebagai `NaN`. Jadi, hasil `isna()` bisa nol meskipun ada sel yang tampak kosong. Kita akan menangani contoh itu pada kolom `TotalCharges`.

In [ ]:
display(df.describe(include="all").T)
print("Missing values per kolom:")
display(df.isna().sum().sort_values(ascending=False))
print("Jumlah baris duplikat:", df.duplicated().sum())
print("Jumlah nilai unik per kolom:")
display(df.nunique().sort_values())

# 3. Visualisasi keseluruhan dataset

Tidak ada satu grafik yang cocok untuk semua kolom. Kolom numerik dan kategorikal punya makna berbeda, sehingga visualisasinya juga berbeda. Kita akan membuat kumpulan grafik yang saling melengkapi.

Sebelum memvisualisasikan `TotalCharges`, ubah kolom tersebut menjadi numerik. Pada data latihan ini, kolom tersebut mungkin berisi string kosong. `pd.to_numeric(..., errors="coerce")` mengubah nilai numerik menjadi angka dan nilai yang gagal dikonversi menjadi `NaN`.

**Mengapa `errors="coerce"`?** Supaya satu nilai teks yang tidak valid tidak membuat seluruh konversi gagal. Nilai yang menjadi `NaN` harus diperiksa dan ditangani pada tahap preprocessing.

In [ ]:
df["TotalCharges"] = pd.to_numeric(df["TotalCharges"], errors="coerce")

print("Tipe data TotalCharges:", df["TotalCharges"].dtype)
print("TotalCharges yang menjadi missing:", df["TotalCharges"].isna().sum())

## 3A. Pisahkan kolom numerik dan kategorikal

`select_dtypes()` memilih kolom berdasarkan tipe data.

- `include="number"` memilih kolom numerik.
- `include=["object", "category", "bool"]` memilih kolom yang umumnya kategorikal.
- `.columns.tolist()` mengubah daftar nama kolom menjadi Python list.

Kita melakukan ini agar kode visualisasi bisa berjalan otomatis walaupun jumlah kolom berubah. Karena `Churn` adalah kategori, kolom tersebut masuk ke kelompok kategorikal.

In [ ]:
numeric_cols = df.select_dtypes(include="number").columns.tolist()
categorical_cols = df.select_dtypes(
    include=["object", "category", "bool"]
).columns.tolist()

print("Kolom numerik:", numeric_cols)
print("Kolom kategorikal:", categorical_cols)

## 3B. Histogram untuk seluruh kolom numerik

Histogram memperlihatkan bagaimana nilai sebuah kolom numerik tersebar.

- `df[numeric_cols]` memilih semua kolom numerik.
- `.hist(bins=20)` membuat histogram untuk setiap kolom numerik; `bins=20` membagi rentang nilai menjadi sekitar 20 kelompok.
- `figsize=(16, 12)` menentukan ukuran keseluruhan gambar.

**Cara membaca:** sumbu X menunjukkan rentang nilai, sumbu Y menunjukkan banyaknya baris di rentang tersebut. Histogram bisa membantu menemukan distribusi miring, penumpukan nilai, atau nilai ekstrem. Histogram bukan bukti bahwa suatu nilai salah.

In [ ]:
df[numeric_cols].hist(
    bins=20,
    figsize=(16, 12),
    edgecolor="black"
)
plt.suptitle("Distribusi Semua Kolom Numerik", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

## 3C. Bar chart untuk seluruh kolom kategorikal

Bar chart ini dibuat satu per satu menggunakan `for`.

- `for col in categorical_cols:` berarti jalankan blok kode untuk setiap nama kolom kategorikal.
- `value_counts()` menghitung jumlah masing-masing kategori.
- `.head(15)` membatasi grafik ke 15 kategori teratas supaya tidak terlalu padat.
- `sns.barplot(x=counts.values, y=counts.index)` menggambar jumlah kategori secara horizontal.

**Kenapa tidak memakai histogram?** Kategori seperti `Contract` atau `InternetService` bukan angka kontinu. Kita ingin membandingkan jumlah masing-masing label.

In [ ]:
for col in categorical_cols:
    counts = df[col].value_counts().head(15)

    plt.figure(figsize=(8, 4))
    sns.barplot(x=counts.values, y=counts.index)

    plt.title(f"Distribusi {col}")
    plt.xlabel("Jumlah baris")
    plt.ylabel(col)
    plt.tight_layout()
    plt.show()

## 3D. Distribusi target: apakah kelas seimbang?

`value_counts()` menghitung jumlah `Yes` dan `No`. `normalize=True` menghitung proporsi; dikali 100 menjadi persentase.

Count plot menampilkan jumlah per kelas. Ini penting karena jika satu kelas jauh lebih banyak, model bisa terlihat bagus berdasarkan accuracy tetapi buruk dalam menemukan kelas minoritas.

**Yang perlu diamati:** bandingkan tinggi batang dan persentase kelas. Jangan menyimpulkan dataset seimbang atau tidak hanya dari accuracy model.

In [ ]:
class_counts = df["Churn"].value_counts()
class_percent = (df["Churn"].value_counts(normalize=True) * 100).round(2)

print("Jumlah kelas:")
display(class_counts)
print("Persentase kelas (%):")
display(class_percent)

plt.figure(figsize=(7, 4))
ax = sns.countplot(data=df, x="Churn", order=class_counts.index)
ax.set_title("Distribusi Target Churn")
ax.set_xlabel("Churn")
ax.set_ylabel("Jumlah pelanggan")
for container in ax.containers:
    ax.bar_label(container)
plt.tight_layout()
plt.show()

## 3E. Boxplot: membandingkan angka berdasarkan kategori

Boxplot berikut membandingkan `MonthlyCharges` antara pelanggan yang churn dan tidak.

- `x="Churn"` membuat kelompok di sumbu X.
- `y="MonthlyCharges"` menunjukkan sebaran nilai numerik di tiap kelompok.
- Garis di dalam kotak adalah median.
- Kotak menunjukkan rentang kuartil pertama sampai kuartil ketiga.
- Titik di luar whisker biasanya ditandai sebagai kandidat outlier.

**Logika analisis:** apakah median atau sebaran biaya bulanan terlihat berbeda antar kelompok? Grafik ini menunjukkan asosiasi/deskripsi, bukan bukti bahwa biaya bulanan menyebabkan churn.

In [ ]:
plt.figure(figsize=(8, 5))
sns.boxplot(data=df, x="Churn", y="MonthlyCharges")
plt.title("Monthly Charges berdasarkan Churn")
plt.xlabel("Churn")
plt.ylabel("Monthly Charges")
plt.tight_layout()
plt.show()

## 3F. Heatmap korelasi numerik

`.corr()` menghitung korelasi antar kolom numerik. Heatmap memetakan nilai tersebut ke warna agar pola lebih mudah dilihat.

- `annot=True` menampilkan angka korelasi.
- `fmt=".2f"` menampilkan dua angka di belakang koma.
- `cmap="coolwarm"` memilih palet warna.
- Korelasi mendekati `1` berarti hubungan linear positif; mendekati `-1` berarti negatif; mendekati `0` berarti hubungan linear lemah.

**Penting:** korelasi tidak membuktikan sebab-akibat. Korelasi Pearson juga tidak selalu menangkap hubungan non-linear.

In [ ]:
corr = df[numeric_cols].corr()

plt.figure(figsize=(10, 7))
sns.heatmap(corr, annot=True, cmap="coolwarm", fmt=".2f", vmin=-1, vmax=1)
plt.title("Correlation Heatmap — Kolom Numerik")
plt.tight_layout()
plt.show()

## 3G. Missing values secara visual

Kita menghitung nilai kosong per kolom lalu memvisualisasikan hanya kolom yang memiliki nilai kosong.

- `df.isna().sum()` menghitung jumlah `NaN`.
- `missing[missing > 0]` menyaring kolom yang jumlah missing-nya lebih dari nol.
- Bar chart memudahkan kita melihat kolom mana yang paling perlu diperiksa.

Jika tidak ada missing value yang terdeteksi, bukan berarti tidak ada string kosong atau nilai placeholder seperti `"Unknown"`. Perlu memahami cara data direpresentasikan.

In [ ]:
missing = df.isna().sum().sort_values(ascending=False)
missing = missing[missing > 0]

if len(missing) > 0:
    plt.figure(figsize=(9, 4))
    sns.barplot(x=missing.values, y=missing.index)
    plt.title("Missing Values per Kolom")
    plt.xlabel("Jumlah missing values")
    plt.ylabel("Kolom")
    plt.tight_layout()
    plt.show()
else:
    print("Tidak ada NaN yang terdeteksi.")

## 3H. Satu dashboard ringkas

Contoh ini menempatkan empat pertanyaan EDA ke dalam satu gambar: distribusi `tenure`, jumlah kelas churn, boxplot biaya bulanan, dan korelasi numerik.

- `plt.subplots(2, 2)` membuat kisi 2 baris × 2 kolom.
- `axes[0, 0]`, `axes[0, 1]`, dan seterusnya mengacu pada panel.
- Parameter `ax=...` memberi tahu Seaborn panel mana yang harus diisi.
- `tight_layout()` mengurangi tumpang tindih label.

Dashboard ringkas berguna untuk laporan, tetapi untuk pemeriksaan mendalam kita tetap bisa melihat setiap grafik secara terpisah.

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 10))

axes[0, 0].hist(df["tenure"].dropna(), bins=20, edgecolor="black")
axes[0, 0].set_title("Distribusi Tenure")
axes[0, 0].set_xlabel("Bulan")
axes[0, 0].set_ylabel("Jumlah")

sns.countplot(data=df, x="Churn", ax=axes[0, 1])
axes[0, 1].set_title("Distribusi Churn")

sns.boxplot(data=df, x="Churn", y="MonthlyCharges", ax=axes[1, 0])
axes[1, 0].set_title("Monthly Charges per Churn")

sns.heatmap(df[numeric_cols].corr(), cmap="coolwarm", vmin=-1, vmax=1, ax=axes[1, 1])
axes[1, 1].set_title("Korelasi Numerik")

plt.suptitle("Ringkasan EDA Dataset Telco", fontsize=16)
plt.tight_layout()
plt.show()

# 3I. LAB PREPROCESSING LENGKAP — membersihkan data mentah sebelum modeling

**Bagian tambahan untuk jaga-jaga ETS.** Di sini kita belajar penamaan kolom, merapikan teks, format angka koma/titik, tanggal, duplikat, missing values, outlier, encoding, scaling, feature engineering, feature selection, discretization, dan PCA.

**Dua tahap yang berbeda:**
1. **Pembersihan deterministik** (misalnya mengganti nama kolom, membetulkan pemisah desimal, menyamakan ejaan kategori) bisa dilakukan pada seluruh data jika aturannya tidak *dipelajari* dari distribusi data.
2. **Transformasi yang mempelajari statistik data** (imputasi median, scaling, feature selection, PCA, encoding kategori yang di-fit) harus di-`fit` hanya pada data train atau di dalam pipeline CV. Ini mencegah *data leakage*.

Contoh-contoh awal memakai `demo` atau `df_clean` sebagai salinan agar **notebook asli dan pipeline model tetap aman**. Jika ingin memakai hasil pembersihan untuk model, nanti ada cell integrasinya.

## A. Perbaiki nama kolom (rename, strip, snake_case)

Kadang dataset punya kolom ` Monthly Charges `, `Customer ID`, atau `TOTAL-Charges`. Nama seperti ini merepotkan saat dipanggil dalam kode.

- `.str.strip()` menghapus spasi di awal/akhir.
- `.str.lower()` membuat semua huruf kecil.
- `.str.replace(r"[^a-z0-9]+", "_", regex=True)` mengganti rangkaian karakter selain huruf/angka menjadi `_`.
- `.str.strip("_")` menghapus `_` berlebih di pinggir.
- `.rename(columns={...})` cocok bila hanya ingin mengganti beberapa kolom.

**Peringatan:** jangan menerapkan rename global pada `df` utama di tengah notebook, karena cell selanjutnya mengacu ke `MonthlyCharges`, `Churn`, dan nama asli lainnya.

In [ ]:
import re

demo_nama = pd.DataFrame({
    " Customer ID ": ["C01", "C02"],
    "Monthly Charges (Rp)": [100000, 120000],
    "Churn Status": ["No", "Yes"]
})

def clean_column_name(name):
    name = str(name).strip().lower()
    name = re.sub(r"[^a-z0-9]+", "_", name)
    return name.strip("_")

print("Sebelum:", demo_nama.columns.tolist())
demo_nama.columns = [clean_column_name(c) for c in demo_nama.columns]
print("Sesudah:", demo_nama.columns.tolist())

# Alternatif: ganti satu nama saja
print(demo_nama.rename(columns={"customer_id": "id_pelanggan"}).columns.tolist())

## B. Membersihkan isi teks dan menyamakan label kategori

Nilai `" yes "`, `"YES"`, `"Y"` mungkin bermakna sama, tetapi komputer menganggapnya berbeda.

- `.astype("string")` membuat operasi string konsisten.
- `.str.strip()` membuang spasi tepi.
- `.str.lower()` menyeragamkan huruf.
- `.replace({...})` memetakan variasi ke satu label baku.

**Logika:** satu makna seharusnya memiliki satu representasi. Tetapi jangan menyamakan kategori yang maknanya belum dipastikan sama.

In [ ]:
demo_teks = pd.Series([" YES ", "yes", "Y", "No", " NO ", "N", None], name="churn_mentah")

hasil_teks = (
    demo_teks.astype("string")
    .str.strip()
    .str.lower()
    .replace({"y": "yes", "n": "no"})
)

print(pd.DataFrame({"sebelum": demo_teks, "sesudah": hasil_teks}))

## C. Angka dengan koma dan titik: format Indonesia vs internasional

Ini sering muncul pada CSV atau Excel:

- **Indonesia:** `1.234,56` artinya seribu dua ratus tiga puluh empat koma lima enam.
- **Internasional:** `1,234.56` punya nilai yang sama.
- `1,5` bisa berarti desimal satu koma lima, **bukan** seribu lima ratus.

**Urutan penting:** bila format Indonesia, hapus `.` sebagai pemisah ribuan **dulu**, lalu ubah `,` menjadi `.` sebagai desimal. `pd.to_numeric(errors="coerce")` mengubah teks tak valid menjadi `NaN`, jadi cek hasilnya.

**Jangan langsung menjalankan kedua penggantian tanpa mengetahui format sumber.** Untuk CSV berformat Indonesia, `pd.read_csv(..., sep=";", decimal=",", thousands=".")` juga bisa dipakai saat membaca file, jika separator dan formatnya memang demikian.

In [ ]:
demo_angka = pd.DataFrame({
    "angka_id": ["1.234,56", "25,50", "3.000", "-12,75", "tidak ada"],
    "angka_us": ["1,234.56", "25.50", "3,000", "-12.75", "N/A"]
})

def parse_indonesia(series):
    teks = series.astype("string").str.strip()
    teks = teks.str.replace(".", "", regex=False)
    teks = teks.str.replace(",", ".", regex=False)
    return pd.to_numeric(teks, errors="coerce")

def parse_international(series):
    teks = series.astype("string").str.strip()
    teks = teks.str.replace(",", "", regex=False)
    return pd.to_numeric(teks, errors="coerce")

demo_angka["hasil_id"] = parse_indonesia(demo_angka["angka_id"])
demo_angka["hasil_us"] = parse_international(demo_angka["angka_us"])
display(demo_angka)

## D. CSV berseparator koma vs titik koma

Pemisah **antar kolom** dan pemisah **desimal** adalah dua hal berbeda. Jika CSV memakai `;` sebagai pemisah kolom dan `,` sebagai desimal, gunakan:

`pd.read_csv("nama.csv", sep=";", decimal=",", thousands=".")`

Kalau file menggunakan `,` untuk memisahkan kolom dan `.` untuk desimal, biasanya cukup `pd.read_csv("nama.csv")`.

**Ciri separator salah:** semua isi baris terbaca sebagai satu kolom panjang. Periksa `df.shape` dan `df.head()` segera setelah membaca CSV.

In [ ]:
from io import StringIO

contoh_csv = """nama;harga;jumlah
A;1.250,50;2
B;750,25;3
"""
demo_csv = pd.read_csv(StringIO(contoh_csv), sep=";", decimal=",", thousands=".")
display(demo_csv)
print(demo_csv.dtypes)

## E. Missing values: deteksi, normalisasi placeholder, dan imputasi

Data kosong bisa berupa `NaN`, string kosong `""`, atau teks seperti `"?"`, `"unknown"`, `"N/A"`.

- `.replace([...], np.nan)` menormalkan placeholder menjadi missing yang nyata.
- `.isna().sum()` menghitungnya.
- `.fillna(...)` mengisi missing. Untuk contoh kecil, bisa menggunakan konstanta.
- Pada **modeling**, median/modus harus dipelajari dari **train** dengan `SimpleImputer` di dalam pipeline (sudah ada pada bagian 5 notebook).

**Jangan otomatis menghapus semua baris missing:** pertimbangkan persentase, penyebab missing, dan pengaruhnya pada target.

In [ ]:
demo_missing = pd.DataFrame({
    "usia": [20, np.nan, 24, 22],
    "kota": ["Surabaya", "?", "", "Malang"]
})

demo_missing["kota"] = demo_missing["kota"].replace(["?", "", "unknown"], np.nan)
print("Missing sebelum penanganan:")
display(demo_missing.isna().sum())

# Demonstrasi saja: median/modus dihitung dari data contoh.
demo_missing["usia"] = demo_missing["usia"].fillna(demo_missing["usia"].median())
demo_missing["kota"] = demo_missing["kota"].fillna(demo_missing["kota"].mode()[0])
display(demo_missing)

## F. Duplikat dan pemeriksaan ID

`duplicated()` mendeteksi **baris yang seluruh nilainya sama**. Namun dua baris dengan `customerID` sama tetapi isi berbeda juga patut diperiksa.

- `df.duplicated().sum()` = jumlah duplikat seluruh baris.
- `df.duplicated(subset=["customerID"]).sum()` = jumlah ID berulang.
- `.drop_duplicates()` menghapus duplikat; gunakan hanya setelah memahami apakah pengulangan memang kesalahan. Data transaksi bisa sah memiliki beberapa baris untuk satu pelanggan.

In [ ]:
print("Duplikat seluruh baris:", df.duplicated().sum())
if "customerID" in df.columns:
    print("ID pelanggan berulang:", df.duplicated(subset=["customerID"]).sum())

# Contoh menghapus baris identik, TANPA mengubah df utama:
demo_tanpa_duplikat = df.drop_duplicates().copy()
print("Ukuran sebelum/sesudah:", df.shape, demo_tanpa_duplikat.shape)

## G. Parsing tanggal dan feature engineering dari tanggal

`pd.to_datetime()` mengubah teks tanggal menjadi tipe datetime. `dayfirst=True` berguna untuk format `DD/MM/YYYY`; untuk format yang konsisten, parameter `format=` lebih aman.

Dari tanggal, kita bisa membuat fitur `bulan`, `hari_dalam_pekan`, atau `lama_hari`.

**Penting:** fitur yang menggunakan tanggal acuan harus memakai tanggal yang tersedia pada saat prediksi, bukan tanggal masa depan. Kalau data merupakan deret waktu, gunakan pembagian train/test berdasarkan waktu agar tidak terjadi leakage.

In [ ]:
demo_tanggal = pd.DataFrame({"tanggal_daftar": ["01/02/2026", "15/03/2026", "29/04/2026"]})
demo_tanggal["tanggal_daftar"] = pd.to_datetime(
    demo_tanggal["tanggal_daftar"], format="%d/%m/%Y", errors="coerce"
)
demo_tanggal["bulan_daftar"] = demo_tanggal["tanggal_daftar"].dt.month
demo_tanggal["hari_ke"] = demo_tanggal["tanggal_daftar"].dt.dayofweek

# Tanggal acuan tetap agar contoh reproducible
tanggal_acuan = pd.Timestamp("2026-05-01")
demo_tanggal["hari_sejak_daftar"] = (tanggal_acuan - demo_tanggal["tanggal_daftar"]).dt.days
display(demo_tanggal)

## H. Outlier: IQR, boxplot, clipping

IQR = Q3 − Q1. Aturan umum memberi batas bawah `Q1 − 1.5×IQR` dan batas atas `Q3 + 1.5×IQR`.

- Outlier **bukan otomatis error**; bisa merupakan pelanggan nyata dengan tagihan sangat tinggi.
- `.clip(lower, upper)` membatasi nilai ekstrem tanpa menghapus baris.
- Jika batas outlier dipakai untuk modeling, hitung batas **dari train** lalu terapkan batas yang sama pada test.

Contoh di bawah hanya *eksplorasi*, bukan perubahan pada dataset modeling.

In [ ]:
kolom = "MonthlyCharges"
q1 = df[kolom].quantile(0.25)
q3 = df[kolom].quantile(0.75)
iqr = q3 - q1
batas_bawah = q1 - 1.5 * iqr
batas_atas = q3 + 1.5 * iqr

print("Batas IQR:", batas_bawah, batas_atas)
print("Jumlah kandidat outlier:", ((df[kolom] < batas_bawah) | (df[kolom] > batas_atas)).sum())

fig, ax = plt.subplots(1, 2, figsize=(11, 4))
sns.boxplot(y=df[kolom], ax=ax[0])
ax[0].set_title("Sebelum clipping")
sns.boxplot(y=df[kolom].clip(batas_bawah, batas_atas), ax=ax[1])
ax[1].set_title("Sesudah clipping (simulasi)")
plt.tight_layout()
plt.show()

## I. Feature engineering: membuat fitur baru yang masuk akal

Feature engineering berarti membuat fitur yang diharapkan membantu model menemukan pola. Contoh dari Telco Churn:

- `charges_per_month_est` = `TotalCharges / tenure` (dengan perlindungan pembagian nol).
- `is_new_customer` = 1 jika `tenure` ≤ 6 bulan.
- `has_internet` = 1 jika `InternetService` bukan `No`.

**Logika:** fitur baru merangkum informasi yang relevan dengan persoalan. Namun fitur baru bisa redundan, dan tidak selalu meningkatkan skor. Evaluasi dengan cross-validation. Jangan gunakan informasi yang belum tersedia saat prediksi (target leakage).

In [ ]:
df_fe = df.copy()

df_fe["charges_per_month_est"] = (
    df_fe["TotalCharges"] / df_fe["tenure"].replace(0, np.nan)
)
df_fe["is_new_customer"] = (df_fe["tenure"] <= 6).astype(int)
df_fe["has_internet"] = (df_fe["InternetService"] != "No").astype(int)

display(df_fe[[
    "tenure", "TotalCharges", "charges_per_month_est",
    "is_new_customer", "has_internet"
]].head(10))

# Perhatikan: df utama tetap utuh. df_fe adalah salinan dengan fitur tambahan.

## J. Discretization / binning: angka menjadi kelompok

Binning mengubah variabel numerik menjadi kategori rentang. Misalnya `tenure` menjadi `baru`, `menengah`, `lama`.

- `pd.cut()` menggunakan batas angka yang kita tentukan.
- `pd.qcut()` membuat kelompok berdasarkan kuantil data.

**Perbedaan penting:** batas `pd.cut` yang ditetapkan dari aturan bisnis dapat konsisten dipakai di train/test. `qcut` mempelajari kuantil dari data; untuk modeling, kuantil harus dihitung dari train saja. Binning bisa mempermudah interpretasi, tetapi mengurangi detail informasi angka.

In [ ]:
df_binning = df[["tenure", "Churn"]].copy()
df_binning["kelompok_tenure"] = pd.cut(
    df_binning["tenure"],
    bins=[-1, 6, 24, np.inf],
    labels=["baru", "menengah", "lama"]
)
display(df_binning.head())
print(df_binning["kelompok_tenure"].value_counts())

sns.countplot(data=df_binning, x="kelompok_tenure", hue="Churn")
plt.title("Kelompok tenure berdasarkan churn")
plt.tight_layout()
plt.show()

## K. Encoding kategori: label mapping vs one-hot

Model numerik perlu merepresentasikan kategori dalam bentuk angka.

- **Mapping** cocok untuk kategori berurutan yang punya makna, misalnya `rendah=0`, `sedang=1`, `tinggi=2`.
- **One-hot encoding** membuat kolom indikator terpisah, cocok untuk kategori nominal seperti `Contract`.
- Jangan memberi kode `0,1,2` pada kategori nominal tanpa alasan; beberapa model dapat menganggapnya memiliki urutan.

Untuk modeling kita sudah memakai `OneHotEncoder(handle_unknown="ignore")` dalam pipeline. Di sini kita lihat versi pandas untuk memahami hasilnya.

In [ ]:
demo_encoding = pd.DataFrame({
    "prioritas": ["rendah", "sedang", "tinggi"],
    "kontrak": ["Month-to-month", "One year", "Two year"]
})

demo_encoding["prioritas_ordinal"] = demo_encoding["prioritas"].map({
    "rendah": 0, "sedang": 1, "tinggi": 2
})
demo_ohe = pd.get_dummies(demo_encoding, columns=["kontrak"], dtype=int)
display(demo_ohe)

## L. Scaling: StandardScaler vs MinMaxScaler

**Mengapa scaling?** Algoritma berbasis jarak seperti KNN dan SVM dengan kernel tertentu bisa lebih dipengaruhi fitur yang memiliki rentang nilai besar.

- `StandardScaler`: mengubah nilai menjadi `(x − mean) / std`, sehingga rata-rata mendekati 0 dan standar deviasi mendekati 1 pada data train.
- `MinMaxScaler`: mengubah rentang train menjadi sekitar 0–1. Nilai test dapat berada di luar 0–1 jika melewati rentang train.
- Decision Tree/Random Forest biasanya tidak memerlukan scaling.

**Fit hanya pada train:** scaler mempelajari mean/std atau min/max dari train; gunakan `.transform()` pada test, jangan `.fit_transform()` ulang.

In [ ]:
from sklearn.preprocessing import StandardScaler, MinMaxScaler

contoh_train = pd.DataFrame({"biaya": [20.0, 40.0, 80.0, 100.0]})
contoh_test = pd.DataFrame({"biaya": [120.0]})

standard = StandardScaler().fit(contoh_train)
minmax = MinMaxScaler().fit(contoh_train)

print("StandardScaler train:", standard.transform(contoh_train).ravel())
print("StandardScaler test:", standard.transform(contoh_test).ravel())
print("MinMaxScaler train:", minmax.transform(contoh_train).ravel())
print("MinMaxScaler test:", minmax.transform(contoh_test).ravel())

## O. Latihan code completion — kerjakan tanpa melihat contoh di atas

Isi titik kosong ini di cell latihan **setelah membaca konsepnya**:

1. Hapus spasi pada teks kategori: `df["kolom"].str.____()`
2. Ubah string `"1.250,75"` format Indonesia menjadi angka: hapus `"."` lalu ganti `","` menjadi `"."`.
3. Cek missing value tiap kolom: `df.____().sum()`
4. Ubah teks tanggal ke datetime: `pd.____(kolom, format="%d/%m/%Y")`
5. Hitung Q1: `df["tenure"].____(0.25)`
6. Pisahkan train/test dengan `stratify=____`.
7. Dalam pipeline, apakah `fit_transform` scaler boleh dipanggil langsung pada test? Jelaskan alasannya.

**Tujuan latihan:** bukan hafal semua sintaks, melainkan mengenali jenis masalah data dan memilih transformasi yang tepat.

In [ ]:
# Isi kode latihan di bawah ini sendiri.
# Contoh:
# df_latihan = df.copy()
# df_latihan["gender"] = df_latihan["gender"].str.____()

# 4. Persiapan fitur dan target

Model belajar dari fitur (`X`) untuk memprediksi target (`y`).

- `Churn` adalah target yang ingin diprediksi, jadi tidak boleh ikut masuk ke fitur.
- `customerID` hanya pengenal pelanggan; pada latihan ini kita keluarkan agar model tidak mencoba mempelajari ID.
- `drop(columns=[...])` menghapus kolom yang disebutkan.
- `train_test_split` memisahkan data untuk pelatihan dan pengujian.

`test_size=0.2` berarti 20% data menjadi test set. `stratify=y` berusaha mempertahankan proporsi kelas di train dan test. `random_state=42` membuat pembagian dapat diulang.

**Pencegahan leakage:** test set disimpan untuk evaluasi akhir. Jangan gunakan test set untuk memilih parameter atau melakukan oversampling.

In [ ]:
from sklearn.model_selection import train_test_split

X = df.drop(columns=["Churn", "customerID"])
y = df["Churn"]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)
print("Proporsi kelas train:")
display(y_train.value_counts(normalize=True).round(3))
print("Proporsi kelas test:")
display(y_test.value_counts(normalize=True).round(3))

# 5. Preprocessing di dalam Pipeline

Model scikit-learn membutuhkan fitur yang sudah siap digunakan. Dataset ini memiliki numerik dan kategorikal, sehingga keduanya diproses dengan cara berbeda.

**Kolom numerik**
1. `SimpleImputer(strategy="median")` mengisi nilai kosong dengan median yang dipelajari dari data train.
2. `StandardScaler()` menstandarkan skala numerik. Untuk Random Forest, scaling biasanya tidak wajib, tetapi ditampilkan untuk belajar preprocessing umum.

**Kolom kategorikal**
1. `SimpleImputer(strategy="most_frequent")` mengisi missing dengan kategori yang paling sering muncul.
2. `OneHotEncoder(handle_unknown="ignore")` mengubah kategori menjadi fitur biner; kategori baru saat test tidak menyebabkan error.

`ColumnTransformer` menerapkan transformasi yang berbeda pada kolom yang berbeda. `Pipeline` menggabungkan preprocessing dan model menjadi satu alur.

**Kenapa fit setelah split?** Imputer, scaler, dan encoder harus belajar dari train set saja. Jika mereka mempelajari seluruh data sebelum split, informasi test dapat bocor ke proses training.

In [ ]:
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

numeric_features = X_train.select_dtypes(include="number").columns.tolist()
categorical_features = X_train.select_dtypes(exclude="number").columns.tolist()

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numeric_features),
    ("cat", categorical_transformer, categorical_features)
])

rf_pipeline = Pipeline(steps=[
    ("preprocess", preprocessor),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

rf_pipeline.fit(X_train, y_train)
print("Model selesai dilatih.")

# 6. Evaluasi model

Setelah training, kita menguji model pada test set yang belum dipakai untuk melatih model.

- `predict(X_test)` menghasilkan label prediksi (`Yes`/`No`).
- `accuracy` adalah proporsi semua prediksi yang benar.
- `precision` menjawab: dari semua prediksi `Yes`, berapa yang benar-benar `Yes`?
- `recall` menjawab: dari semua pelanggan yang sebenarnya `Yes`, berapa yang berhasil ditemukan?
- `F1` menggabungkan precision dan recall melalui harmonic mean.
- `confusion_matrix` merangkum prediksi benar dan salah.
- `classification_report` menampilkan metrik per kelas.

Karena target positif kita adalah `Yes`, kita menyebut `pos_label="Yes"` pada metrik biner. ROC-AUC menggunakan probabilitas kelas positif, bukan hanya label prediksi.

In [ ]:
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report, roc_auc_score
)

y_pred = rf_pipeline.predict(X_test)
y_prob = rf_pipeline.predict_proba(X_test)[:, 1]

print("Accuracy:", accuracy_score(y_test, y_pred))
print("Precision (Yes):", precision_score(y_test, y_pred, pos_label="Yes"))
print("Recall (Yes):", recall_score(y_test, y_pred, pos_label="Yes"))
print("F1 (Yes):", f1_score(y_test, y_pred, pos_label="Yes"))
print("ROC-AUC:", roc_auc_score((y_test == "Yes").astype(int), y_prob))
print("\nClassification report:")
print(classification_report(y_test, y_pred))

cm = confusion_matrix(y_test, y_pred, labels=["No", "Yes"])
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues",
            xticklabels=["Prediksi No", "Prediksi Yes"],
            yticklabels=["Aktual No", "Aktual Yes"])
plt.title("Confusion Matrix")
plt.tight_layout()
plt.show()

## M. Feature selection dan PCA (setelah train/test split)

Feature selection memilih fitur yang dianggap berguna, sedangkan PCA membuat kombinasi baru dari fitur numerik untuk merangkum variasi.

- `SelectKBest(f_classif, k=2)` memilih dua fitur berdasarkan uji hubungan univariat dengan target.
- `PCA(n_components=2)` memproyeksikan fitur numerik yang sudah distandardisasi menjadi dua komponen.

**Keduanya belajar dari data:** fit hanya pada train. PCA bukan pemilihan dua kolom terbaik; komponennya merupakan kombinasi linear. PCA tidak otomatis meningkatkan skor klasifikasi. Untuk dataset campuran, one-hot dan imputasi harus dipertimbangkan sebelum metode ini.

In [ ]:
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler

fitur_numerik = ["tenure", "MonthlyCharges", "TotalCharges"]
X_num_train = X_train[fitur_numerik]
X_num_test = X_test[fitur_numerik]

# Imputasi dan scaling di-fit hanya pada train
num_imputer = SimpleImputer(strategy="median")
num_scaler = StandardScaler()
X_num_train_ready = num_scaler.fit_transform(num_imputer.fit_transform(X_num_train))
X_num_test_ready = num_scaler.transform(num_imputer.transform(X_num_test))

selector = SelectKBest(score_func=f_classif, k=2)
selector.fit(X_num_train_ready, y_train)
print("Fitur terpilih:", list(pd.Index(fitur_numerik)[selector.get_support()]))
print("Ukuran test setelah selection:", selector.transform(X_num_test_ready).shape)

pca = PCA(n_components=2, random_state=42)
train_pca = pca.fit_transform(X_num_train_ready)
test_pca = pca.transform(X_num_test_ready)
print("Proporsi variasi yang dijelaskan PCA:", pca.explained_variance_ratio_)

sns.scatterplot(x=train_pca[:, 0], y=train_pca[:, 1], hue=y_train.to_numpy(), alpha=0.6)
plt.title("PCA 2D (data train saja)")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.tight_layout()
plt.show()

## N. Integrasi feature engineering ke Pipeline (cara aman untuk eksperimen)

Jika fitur baru dihitung dari **kolom pada baris yang sama** dan tidak menggunakan target atau statistik seluruh dataset, kita dapat membuat fungsi transformasi deterministik. Dengan `FunctionTransformer`, fungsi ini menjadi langkah pertama pipeline sebelum imputer, encoder, dan model.

**Keuntungan:** transformasi yang sama otomatis diterapkan saat `.fit()` dan `.predict()`; tidak perlu mengubah test secara manual.

Cell ini adalah contoh **mandiri** yang memakai `X_train` dan `X_test` dari bagian 4. Jalankan setelah bagian 4–5 jika belum dijalankan. Untuk menjaga kompatibilitas antarversi, kita gunakan `validate=False` agar input tetap DataFrame.

In [ ]:
from sklearn.preprocessing import FunctionTransformer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier

def add_telco_features(frame):
    out = frame.copy()
    out["is_new_customer"] = (out["tenure"] <= 6).astype(int)
    out["has_internet"] = (out["InternetService"] != "No").astype(int)
    out["charges_per_month_est"] = (
        out["TotalCharges"] / out["tenure"].replace(0, np.nan)
    )
    return out

# Tentukan kolom hasil transformasi dari struktur train (tanpa mempelajari statistik).
feature_preview = add_telco_features(X_train)
num_fe = feature_preview.select_dtypes(include="number").columns.tolist()
cat_fe = feature_preview.select_dtypes(exclude="number").columns.tolist()

preprocess_fe = ColumnTransformer([
    ("num", Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ]), num_fe),
    ("cat", Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]), cat_fe)
])

pipeline_fe = Pipeline([
    ("feature_engineering", FunctionTransformer(add_telco_features, validate=False)),
    ("preprocess", preprocess_fe),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

pipeline_fe.fit(X_train, y_train)
print("F1 kelas Yes — baseline:", f1_score(y_test, rf_pipeline.predict(X_test), pos_label="Yes"))
print("F1 kelas Yes — dengan FE:", f1_score(y_test, pipeline_fe.predict(X_test), pos_label="Yes"))
print("Untuk memilih fitur secara adil, bandingkan skor CV di train, bukan berulang kali mengoptimalkan berdasarkan test.")

# 7. Membandingkan Random Forest dan AdaBoost

Ensemble learning menggabungkan beberapa model dasar.

- **Random Forest** melatih banyak decision tree dengan variasi sampel/fitur, lalu menggabungkan prediksi mereka.
- **AdaBoost** membangun rangkaian learner yang memberi perhatian lebih pada contoh yang sebelumnya sulit diprediksi.

Kita menggunakan preprocessing yang sama untuk perbandingan yang lebih adil. `clone(preprocessor)` membuat salinan transformer yang belum di-fit. `F1-score` kelas `Yes` dipakai sebagai metrik perbandingan pada latihan ini.

**Catatan:** skor model dapat berubah tergantung dataset, parameter, dan random seed. Tidak ada algoritma yang selalu paling baik.

In [ ]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.base import clone
from sklearn.metrics import f1_score

models = {
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42)
}

results = []

for name, estimator in models.items():
    pipe = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    score = f1_score(y_test, pred, pos_label="Yes")
    results.append({"Model": name, "F1 Yes": score})

results_df = pd.DataFrame(results).sort_values("F1 Yes", ascending=False)
display(results_df)

sns.barplot(data=results_df, x="Model", y="F1 Yes")
plt.title("Perbandingan F1-score kelas Yes")
plt.ylim(0, 1)
plt.tight_layout()
plt.show()

# 8. Cross-validation

Train/test split hanya memberi satu pembagian data. Cross-validation memberi beberapa evaluasi: pada 5-fold CV, data train dibagi menjadi lima bagian; setiap putaran menggunakan empat bagian untuk training dan satu bagian untuk validasi.

- `StratifiedKFold` berusaha menjaga proporsi kelas pada tiap fold.
- `n_splits=5` berarti lima fold.
- `shuffle=True` mengacak data sebelum pembagian.
- `cross_val_score` menjalankan evaluasi berulang dan mengembalikan skor tiap fold.
- `f1_macro` menghitung F1 untuk setiap kelas lalu mengambil rata-rata tanpa bobot jumlah kelas.

Kita melakukan CV hanya pada `X_train, y_train`. Test set tetap disimpan untuk evaluasi akhir.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.base import clone

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

cv_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

scores = cross_val_score(
    cv_pipeline,
    X_train, y_train,
    cv=cv,
    scoring="f1_macro"
)

print("Skor setiap fold:", scores)
print("Rata-rata:", scores.mean())
print("Standar deviasi:", scores.std())

# 9. Class imbalance dan RandomOverSampler

Jika kelas target tidak seimbang, model mungkin kesulitan menemukan kelas yang lebih jarang. Oversampling menambah contoh kelas minoritas pada data training.

`RandomOverSampler` mengambil ulang contoh dari kelas minoritas secara acak dengan pengembalian. Pada contoh ini, sampler berada di dalam `imblearn.pipeline.Pipeline`, sehingga oversampling dilakukan pada bagian training di setiap pemanggilan `fit`, bukan pada test set.

**Penting:** jangan oversample seluruh dataset sebelum train/test split. Itu dapat membuat salinan atau informasi dari sampel training bocor ke test set. Test set harus tetap mencerminkan distribusi data yang belum diubah.

In [ ]:
# Jika belum terpasang, jalankan di notebook:
# %pip install imbalanced-learn

from imblearn.over_sampling import RandomOverSampler
from imblearn.pipeline import Pipeline as ImbPipeline

print("Distribusi train sebelum resampling:")
display(y_train.value_counts())

imb_pipeline = ImbPipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("sampler", RandomOverSampler(random_state=42)),
    ("model", RandomForestClassifier(n_estimators=100, random_state=42))
])

imb_pipeline.fit(X_train, y_train)
imb_pred = imb_pipeline.predict(X_test)

print("F1 sebelum resampling:",
      f1_score(y_test, rf_pipeline.predict(X_test), pos_label="Yes"))
print("F1 setelah resampling:",
      f1_score(y_test, imb_pred, pos_label="Yes"))

print("Distribusi test (tidak diubah):")
display(y_test.value_counts())

# 10. Hyperparameter tuning dengan GridSearchCV

Hyperparameter adalah pengaturan model yang ditentukan sebelum/dalam proses training, misalnya jumlah pohon atau kedalaman maksimum pohon.

`GridSearchCV` mencoba semua kombinasi parameter yang ditentukan di `param_grid` dan menilai setiap kombinasi menggunakan cross-validation.

- Nama parameter pipeline menggunakan format `nama_step__nama_parameter`, misalnya `model__max_depth`.
- `scoring="f1_macro"` berarti kombinasi dipilih berdasarkan rata-rata F1 antar kelas.
- `n_jobs=-1` meminta scikit-learn menggunakan semua core CPU yang tersedia (jika didukung environment).
- `best_params_` memberi kombinasi terbaik; `best_score_` memberi skor CV terbaik.

Fit tuning hanya pada train set. Setelah kombinasi terbaik ditemukan, gunakan test set satu kali untuk evaluasi akhir.

In [ ]:
from sklearn.model_selection import GridSearchCV

tuning_pipeline = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", RandomForestClassifier(random_state=42))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 20],
    "model__min_samples_split": [2, 5]
}

grid = GridSearchCV(
    estimator=tuning_pipeline,
    param_grid=param_grid,
    cv=cv,
    scoring="f1_macro",
    n_jobs=-1
)

grid.fit(X_train, y_train)

print("Best parameters:", grid.best_params_)
print("Best CV F1 macro:", grid.best_score_)

best_model = grid.best_estimator_
best_pred = best_model.predict(X_test)
print("Test F1 kelas Yes:", f1_score(y_test, best_pred, pos_label="Yes"))

# 11. Checklist pemahaman

Coba jawab dengan kata-katamu sendiri setelah menjalankan notebook:

1. Mengapa kita perlu melihat `shape`, `info()`, missing values, dan duplikat sebelum modeling?
2. Kapan histogram lebih cocok daripada count plot?
3. Apa perbedaan boxplot dan heatmap korelasi?
4. Mengapa `Churn` harus dipisahkan dari `X`?
5. Mengapa imputer dan encoder di-fit hanya pada data train?
6. Apa perbedaan precision dan recall untuk kelas `Yes`?
7. Mengapa oversampling tidak boleh dilakukan pada test set?
8. Apa yang dilakukan `GridSearchCV` dan mengapa ia membutuhkan cross-validation?

**Pola berpikir EDA:** setiap grafik sebaiknya menjawab pertanyaan. Jangan hanya membuat grafik karena bisa; tulis satu atau dua kalimat interpretasi berdasarkan hasil yang benar-benar terlihat.

---
# BAGIAN TAMBAHAN — Seluruh materi klasifikasi yang mungkin keluar di ETS

Berdasarkan daftar materi yang kamu tunjukkan, cakupannya adalah: **konsep dasar klasifikasi & Decision Tree, KNN, Naive Bayes, SVM, ANN, Rule-Based, Model Overfitting, dan Ensemble Methods**.

**Target belajar:** bukan sekadar hafal `fit()` dan `predict()`, tetapi memahami *cara model mengambil keputusan, parameter penting, kapan preprocessing dibutuhkan, dan kapan suatu model kurang cocok*.

Kita tetap memakai `X_train`, `X_test`, `y_train`, `y_test`, serta `preprocessor` yang sudah dibuat pada bab sebelumnya. Jalankan notebook dari awal agar variabel-variabel itu tersedia.

> ANN di sini berarti `MLPClassifier` (jaringan saraf sederhana dari scikit-learn), **bukan deep learning yang wajib kamu pakai saat ujian**. Jika aturan praktikum melarang deep learning, jangan gunakan ANN ketika ujian; pelajari konsepnya untuk teori.

## 12. Konsep dasar klasifikasi — apa yang sebenarnya dipelajari model?

**Klasifikasi** mempelajari hubungan fitur (`X`) dan label (`y`) untuk memprediksi *kelas*. Pada dataset ini labelnya `Churn = Yes/No`.

1. **Training:** model mempelajari pola dari `(X_train, y_train)`.
2. **Prediksi:** model menerima fitur pelanggan yang belum dilihat, lalu mengeluarkan prediksi `Yes`/`No`.
3. **Evaluasi:** bandingkan hasil prediksi dengan `y_test`.

Istilah yang sering tertukar:
- **Parameter model:** sesuatu yang dipelajari dari data, seperti bobot ANN atau titik split pada pohon.
- **Hyperparameter:** pengaturan sebelum training, misalnya `max_depth`, `n_neighbors`, atau `C`.
- **Classification vs regression:** klasifikasi menghasilkan kelas, regresi menghasilkan angka kontinu.

**Contoh logika:** jika model menemukan bahwa pelanggan dengan kontrak bulanan dan tenure pendek lebih sering churn, ia bisa menggunakan informasi itu untuk memprediksi kelas pelanggan baru. Ini pola statistik, bukan kepastian atau sebab-akibat.

## 13. Decision Tree — serangkaian pertanyaan bercabang

Bayangkan kamu menebak churn dengan pertanyaan: “Kontrak bulanan?”, “Tenure pendek?”, lalu akhirnya memilih `Yes` atau `No`. **Decision Tree** membuat aturan bercabang seperti itu secara otomatis.

### Bagaimana pohon memilih pertanyaan?
Pohon mencoba membagi data sehingga label dalam setiap cabang menjadi lebih homogen. Ukuran yang umum adalah:
- **Gini impurity:** `1 - Σ p(k)^2`; nilainya rendah ketika node didominasi satu kelas.
- **Entropy:** `-Σ p(k) log2 p(k)`; juga mengukur ketidakmurnian kelas.

Contoh: node dengan 50% Yes dan 50% No memiliki Gini `1 - (0.5² + 0.5²) = 0.5`. Node 100% Yes memiliki Gini `0`. Split yang bagus mengurangi impurity tertimbang pada anak-anak node.

Parameter yang perlu diingat: `criterion`, `max_depth`, `min_samples_split`, `min_samples_leaf`, `ccp_alpha` (pruning). Pohon terlalu dalam berisiko menghafal training data.

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.base import clone
from sklearn.pipeline import Pipeline

# Kita batasi max_depth supaya pohon lebih mudah dipahami
model_dt = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", DecisionTreeClassifier(max_depth=3, criterion="gini", random_state=42))
])
model_dt.fit(X_train, y_train)

print("Accuracy Decision Tree:", model_dt.score(X_test, y_test))

plt.figure(figsize=(18, 8))
plot_tree(
    model_dt.named_steps["model"],
    max_depth=2,
    filled=True,
    class_names=model_dt.named_steps["model"].classes_,
    fontsize=8
)
plt.title("Visualisasi sebagian Decision Tree (fitur berupa indeks hasil preprocessing)")
plt.show()

**Mengapa grafik pohonnya berisi indeks fitur?** Karena `OneHotEncoder` mengubah satu kolom kategori menjadi beberapa kolom biner. Nama fitur asli tidak lagi satu-banding-satu. Di bawah ini kita ambil nama fitur hasil preprocessing agar pohon lebih mudah dibaca.

In [ ]:
feature_names = model_dt.named_steps["preprocess"].get_feature_names_out()
plt.figure(figsize=(20, 9))
plot_tree(
    model_dt.named_steps["model"],
    feature_names=feature_names,
    class_names=model_dt.named_steps["model"].classes_,
    filled=True, max_depth=2, fontsize=8
)
plt.title("Decision Tree dengan nama fitur hasil preprocessing")
plt.show()

## 14. K-Nearest Neighbors (KNN) — lihat tetangga terdekat

KNN tidak membangun pohon atau persamaan aturan eksplisit. Saat ada pelanggan baru, ia:
1. Menghitung jarak pelanggan baru ke pelanggan dalam data train.
2. Memilih **K** tetangga terdekat.
3. Mengambil kelas mayoritas di antara tetangga itu.

Contoh `K=5`: jika 3 tetangga `Yes` dan 2 `No`, prediksinya `Yes`.

**Mengapa scaling penting?** Jika `tenure` berkisar 0–72 dan `MonthlyCharges` 20–120, perbedaan skala bisa membuat satu fitur mendominasi perhitungan jarak. `StandardScaler` membantu menyetarakan skala numerik.

Parameter: `n_neighbors`, `weights` (`uniform` atau `distance`), `metric` (jarak). K terlalu kecil rentan sensitif terhadap noise; K terlalu besar bisa mengaburkan pola lokal.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

model_knn = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", KNeighborsClassifier(n_neighbors=5))
])
model_knn.fit(X_train, y_train)
print("Accuracy KNN:", model_knn.score(X_test, y_test))
print("5 prediksi pertama:", model_knn.predict(X_test.iloc[:5]))

## 15. Naive Bayes — probabilitas dengan asumsi sederhana

Naive Bayes menggunakan aturan Bayes:

`P(Kelas | Fitur) ∝ P(Fitur | Kelas) × P(Kelas)`

Artinya: peluang pelanggan churn setelah mengetahui fitur-fitur mereka ditentukan dari **peluang awal kelas** dan **seberapa mungkin fitur-fitur tersebut muncul pada kelas itu**.

Disebut *naive* karena model mengasumsikan fitur saling independen **jika kelasnya diketahui**. Asumsi ini sering tidak sepenuhnya benar, tetapi model tetap bisa bekerja cukup baik.

Jenis umum:
- `GaussianNB`: untuk fitur numerik kontinu, dengan asumsi distribusi Gaussian per kelas.
- `MultinomialNB`: lazim untuk hitungan non-negatif, misalnya frekuensi kata.
- `BernoulliNB`: untuk fitur biner.

Di latihan ini kita menggunakan **GaussianNB**. Karena beberapa versi `OneHotEncoder` menghasilkan matriks sparse dan `GaussianNB` mengharapkan array dense, kita menambahkan konversi ke dense. Ini aman untuk dataset latihan kecil, tetapi bisa boros memori untuk dataset besar.

In [ ]:
from sklearn.naive_bayes import GaussianNB
from sklearn.preprocessing import FunctionTransformer
from scipy import sparse

def to_dense(X):
    return X.toarray() if sparse.issparse(X) else X

model_nb = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("dense", FunctionTransformer(to_dense)),
    ("model", GaussianNB())
])
model_nb.fit(X_train, y_train)
print("Accuracy Gaussian Naive Bayes:", model_nb.score(X_test, y_test))

## 16. Support Vector Machine (SVM) — cari batas pemisah

SVM mencoba membuat batas keputusan (*decision boundary*) yang memisahkan kelas dengan **margin** sebesar mungkin. Data yang paling berpengaruh terhadap batas disebut **support vectors**.

- **Linear SVM:** batasnya berupa garis/bidang lurus di ruang fitur.
- **Kernel SVM:** dengan kernel seperti `rbf`, model bisa menangani batas non-linear.

Parameter penting:
- `C`: kompromi antara margin yang lebar dan kesalahan klasifikasi. `C` besar cenderung lebih keras menghukum salah klasifikasi.
- `kernel`: misalnya `linear` atau `rbf`.
- `gamma` untuk `rbf`: mengatur seberapa lokal pengaruh tiap contoh. Nilai terlalu besar bisa overfit.

**Scaling sangat penting** karena SVM bergantung pada jarak dan posisi fitur. Preprocessing yang sudah kita buat mencakup scaling numerik.

In [ ]:
from sklearn.svm import SVC

model_svm = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", SVC(kernel="rbf", C=1.0, gamma="scale", random_state=42))
])
model_svm.fit(X_train, y_train)
print("Accuracy SVM:", model_svm.score(X_test, y_test))

## 17. Artificial Neural Network (ANN) — belajar melalui bobot

ANN sederhana memiliki **input layer**, satu atau beberapa **hidden layer**, dan **output layer**. Setiap neuron menggabungkan masukan dengan bobot, menambahkan bias, lalu menerapkan fungsi aktivasi.

Secara garis besar:
1. **Forward pass:** data bergerak dari input ke output untuk menghasilkan prediksi.
2. **Loss:** model menghitung seberapa salah prediksinya.
3. **Backpropagation:** menghitung bagaimana bobot perlu berubah untuk mengurangi loss.
4. **Optimizer:** memperbarui bobot; proses diulang selama beberapa epoch/iterasi.

`MLPClassifier` adalah implementasi jaringan saraf *multilayer perceptron* dari scikit-learn. `hidden_layer_sizes=(32, 16)` berarti dua hidden layer dengan 32 dan 16 neuron.

**Penting untuk aturan ujian:** jika deep learning dilarang, **jangan gunakan ANN pada praktikum**, walaupun materi teorinya masuk daftar. Cell ini untuk belajar konsep dan eksplorasi di luar ujian.

In [ ]:
from sklearn.neural_network import MLPClassifier

model_ann = Pipeline(steps=[
    ("preprocess", clone(preprocessor)),
    ("model", MLPClassifier(
        hidden_layer_sizes=(32, 16),
        activation="relu",
        max_iter=300,
        early_stopping=True,
        random_state=42
    ))
])
model_ann.fit(X_train, y_train)
print("Accuracy ANN sederhana:", model_ann.score(X_test, y_test))

## 18. Rule-Based — aturan IF–THEN yang kita tulis sendiri

Berbeda dari model yang mempelajari pola secara otomatis, sistem rule-based dapat dibuat dengan aturan eksplisit, misalnya:

> IF `Contract == Month-to-month` AND `tenure < 12` THEN `Churn = Yes`; ELSE `No`.

Aturan ini **hanya contoh buatan untuk belajar sintaks**, bukan aturan bisnis yang terbukti benar. Kelebihannya mudah dijelaskan, tetapi aturan manual bisa terlalu sederhana dan gagal menangkap hubungan rumit.

Perhatikan penggunaan `&` untuk **AND** pada kondisi pandas, dan tanda kurung di setiap kondisi.

In [ ]:
# Aturan demonstrasi: jangan menganggapnya sebagai kebenaran bisnis.
rule_pred = np.where(
    (X_test["Contract"] == "Month-to-month") & (X_test["tenure"] < 12),
    "Yes", "No"
)

from sklearn.metrics import accuracy_score, f1_score
print("Accuracy Rule-Based:", accuracy_score(y_test, rule_pred))
print("F1 kelas Yes Rule-Based:", f1_score(y_test, rule_pred, pos_label="Yes"))

## 19. Model Overfitting dan Underfitting — bedakan belajar pola dan menghafal

- **Underfitting:** model terlalu sederhana, sehingga buruk di train dan test.
- **Overfitting:** model terlalu kompleks, sehingga sangat bagus di train tetapi memburuk di test.
- **Generalization:** kemampuan model bekerja pada data baru.

Contoh: Decision Tree tanpa batas kedalaman dapat membuat cabang yang sangat spesifik terhadap training set. Untuk mengamati ini, kita bandingkan skor train dan test pada berbagai `max_depth`.

**Interpretasi:** jika accuracy train terus naik sementara accuracy test stagnan atau turun, itu tanda kemungkinan overfitting. Jangan pilih kedalaman terbaik dengan terus-menerus mengintip test set pada eksperimen serius; gunakan cross-validation pada train set untuk pemilihan hyperparameter.

In [ ]:
from sklearn.tree import DecisionTreeClassifier

depths = [1, 2, 3, 4, 5, 8, 12, None]
train_scores, test_scores = [], []

for depth in depths:
    pipe = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", DecisionTreeClassifier(max_depth=depth, random_state=42))
    ])
    pipe.fit(X_train, y_train)
    train_scores.append(pipe.score(X_train, y_train))
    test_scores.append(pipe.score(X_test, y_test))

labels = [str(d) for d in depths]
plt.figure(figsize=(9, 5))
plt.plot(labels, train_scores, marker="o", label="Train accuracy")
plt.plot(labels, test_scores, marker="o", label="Test accuracy")
plt.xlabel("max_depth (None = tidak dibatasi)")
plt.ylabel("Accuracy")
plt.title("Ilustrasi Underfitting vs Overfitting pada Decision Tree")
plt.legend()
plt.tight_layout()
plt.show()

## 20. Ensemble Methods — banyak model bekerja bersama

**Ensemble** menggabungkan beberapa learner untuk mendapatkan prediksi yang lebih stabil/akurat.

- **Bagging:** melatih banyak learner pada sampel acak berbeda lalu menggabungkan hasilnya. Contoh: `BaggingClassifier`, Random Forest (juga mengacak subset fitur).
- **Boosting:** learner dibangun secara berurutan agar model berikutnya membantu mengatasi kelemahan sebelumnya. Contoh: AdaBoost, Gradient Boosting.
- **Voting:** menggabungkan prediksi beberapa model berbeda melalui suara mayoritas atau rata-rata probabilitas.
- **Stacking:** prediksi beberapa model menjadi input bagi model lain (*meta-model*).

Random Forest dan AdaBoost sudah dibahas pada bab sebelumnya. Di bawah ini kita tambah contoh **Bagging, Gradient Boosting, Voting, dan Stacking**.

**Perhatian:** voting/stacking perlu model yang kompatibel dan evaluasi yang hati-hati. Stacking melakukan pelatihan tambahan dan bisa lebih lambat.

In [ ]:
from sklearn.ensemble import (
    BaggingClassifier, GradientBoostingClassifier,
    VotingClassifier, StackingClassifier
)
from sklearn.linear_model import LogisticRegression

# Model dasar di dalam Voting/Stacking menerima fitur yang SUDAH dipreproses.
base_estimators = [
    ("dt", DecisionTreeClassifier(max_depth=5, random_state=42)),
    ("knn", KNeighborsClassifier(n_neighbors=7)),
    ("lr", LogisticRegression(max_iter=1000, random_state=42))
]

ensemble_models = {
    "Bagging": BaggingClassifier(n_estimators=30, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(n_estimators=100, random_state=42),
    "Voting": VotingClassifier(estimators=base_estimators, voting="hard"),
    "Stacking": StackingClassifier(
        estimators=base_estimators,
        final_estimator=LogisticRegression(max_iter=1000, random_state=42),
        cv=3
    )
}

ensemble_results = []
for name, estimator in ensemble_models.items():
    pipe = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", estimator)
    ])
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    ensemble_results.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "F1 Yes": f1_score(y_test, pred, pos_label="Yes")
    })

display(pd.DataFrame(ensemble_results).sort_values("F1 Yes", ascending=False))

## 21. Perbandingan seluruh model dalam satu tabel dan grafik

Kita bandingkan model dengan **pembagian train/test yang sama** agar hasil lebih mudah dibaca. Namun ini **perbandingan demonstrasi**, bukan eksperimen ilmiah yang sudah dituning secara adil: hyperparameter tiap model belum dioptimalkan, dan evaluasi berulang pada test set bisa membuat keputusan kita bias.

Yang dihitung:
- `Accuracy`: proporsi prediksi benar keseluruhan.
- `Precision Yes`: dari semua prediksi churn, berapa yang benar-benar churn?
- `Recall Yes`: dari semua churn yang benar-benar terjadi, berapa yang ditemukan?
- `F1 Yes`: keseimbangan precision dan recall untuk kelas churn.

Untuk seleksi model yang lebih benar, gunakan **cross-validation pada train set**, pilih model, lalu evaluasi final satu kali pada test set.

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, accuracy_score

# ANN ikut di tabel sebagai bahan belajar, bukan anjuran untuk ujian yang melarang DL.
comparison_models = {
    "Decision Tree": DecisionTreeClassifier(max_depth=5, random_state=42),
    "KNN": KNeighborsClassifier(n_neighbors=5),
    "Naive Bayes": GaussianNB(),
    "SVM": SVC(kernel="rbf", C=1.0),
    "ANN (teori/latihan)": MLPClassifier(
        hidden_layer_sizes=(32, 16), max_iter=300,
        early_stopping=True, random_state=42
    ),
    "Random Forest": RandomForestClassifier(n_estimators=100, random_state=42),
    "AdaBoost": AdaBoostClassifier(n_estimators=100, random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

rows = []
for name, estimator in comparison_models.items():
    steps = [("preprocess", clone(preprocessor))]
    if name == "Naive Bayes":
        steps.append(("dense", FunctionTransformer(to_dense)))
    steps.append(("model", estimator))
    pipe = Pipeline(steps=steps)
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    rows.append({
        "Model": name,
        "Accuracy": accuracy_score(y_test, pred),
        "Precision Yes": precision_score(y_test, pred, pos_label="Yes", zero_division=0),
        "Recall Yes": recall_score(y_test, pred, pos_label="Yes", zero_division=0),
        "F1 Yes": f1_score(y_test, pred, pos_label="Yes", zero_division=0)
    })

# Tambahkan rule-based sebagai baseline sederhana.
rows.append({
    "Model": "Rule-Based (manual)",
    "Accuracy": accuracy_score(y_test, rule_pred),
    "Precision Yes": precision_score(y_test, rule_pred, pos_label="Yes", zero_division=0),
    "Recall Yes": recall_score(y_test, rule_pred, pos_label="Yes", zero_division=0),
    "F1 Yes": f1_score(y_test, rule_pred, pos_label="Yes", zero_division=0)
})

comparison_df = pd.DataFrame(rows).sort_values("F1 Yes", ascending=False)
display(comparison_df.round(3))

plt.figure(figsize=(11, 6))
sns.barplot(data=comparison_df, x="F1 Yes", y="Model")
plt.xlim(0, 1)
plt.title("Perbandingan Model berdasarkan F1 kelas Churn=Yes")
plt.tight_layout()
plt.show()

## 22. Contoh GridSearchCV untuk Decision Tree, KNN, dan SVM

**Kenapa tidak cukup mencoba satu hyperparameter?** Nilai `max_depth=5`, `n_neighbors=5`, atau `C=1` hanya titik awal. Kita dapat mencari kombinasi yang lebih cocok dengan `GridSearchCV` pada data train.

Parameter dalam pipeline diberi awalan `model__`, contohnya `model__max_depth`. Skor `f1_macro` memberi bobot yang sama pada tiap kelas saat menghitung rata-rata F1.

Agar latihan tidak terlalu lama, grid dibuat kecil. Di kompetisi sungguhan, pilih rentang parameter berdasarkan kebutuhan dan waktu komputasi.

In [ ]:
from sklearn.model_selection import GridSearchCV, StratifiedKFold

small_grids = {
    "Decision Tree": (
        DecisionTreeClassifier(random_state=42),
        {"model__max_depth": [3, 5, 10], "model__min_samples_leaf": [1, 5]}
    ),
    "KNN": (
        KNeighborsClassifier(),
        {"model__n_neighbors": [3, 5, 9], "model__weights": ["uniform", "distance"]}
    ),
    "SVM": (
        SVC(),
        {"model__C": [0.1, 1, 10], "model__kernel": ["linear", "rbf"]}
    )
}

cv_small = StratifiedKFold(n_splits=3, shuffle=True, random_state=42)
for name, (estimator, grid_params) in small_grids.items():
    pipe = Pipeline(steps=[
        ("preprocess", clone(preprocessor)),
        ("model", estimator)
    ])
    search = GridSearchCV(
        pipe, grid_params, cv=cv_small,
        scoring="f1_macro", n_jobs=-1
    )
    search.fit(X_train, y_train)
    print(f"{name}: best CV={search.best_score_:.3f}; params={search.best_params_}")

## 23. Cheat sheet pemilihan model

| Model | Ide utama | Perlu scaling? | Hyperparameter utama | Risiko/hal penting |
|---|---|---|---|---|
| Decision Tree | Aturan bercabang | Biasanya tidak | `max_depth`, `criterion`, `ccp_alpha` | Pohon terlalu dalam bisa overfit |
| KNN | Voting tetangga terdekat | **Ya, penting** | `n_neighbors`, `weights` | Lambat untuk prediksi data besar; sensitif skala |
| GaussianNB | Probabilitas Bayes | Tidak wajib | `var_smoothing` | Asumsi Gaussian dan independensi kondisional |
| SVM | Margin pemisah | **Ya, penting** | `C`, `kernel`, `gamma` | Tuning dan komputasi bisa mahal |
| ANN / MLP | Bobot neuron | **Ya, dianjurkan** | `hidden_layer_sizes`, `alpha` | Sensitif hyperparameter; cek aturan ujian |
| Rule-Based | IF–THEN | Tidak wajib | Aturan manual | Aturan mudah terlalu sederhana |
| Random Forest | Banyak pohon + bagging | Biasanya tidak | `n_estimators`, `max_depth` | Kurang mudah dijelaskan dibanding satu pohon |
| AdaBoost | Boosting bertahap | Tergantung learner | `n_estimators`, `learning_rate` | Sensitif terhadap noise pada beberapa kondisi |
| Gradient Boosting | Memperbaiki loss bertahap | Biasanya tidak untuk pohon | `learning_rate`, `n_estimators`, `max_depth` | Bisa overfit jika tidak dituning |
| Voting / Stacking | Menggabungkan model | Tergantung model dasar | Estimator, metode voting | Kompleksitas dan biaya komputasi |

### Mini-latihan code completion

Isi bagian `____` tanpa melihat cell di atas:

```python
# A. KNN dengan 7 tetangga
knn = KNeighborsClassifier(n_neighbors=____)

# B. Decision Tree dengan maksimal kedalaman 4
dt = DecisionTreeClassifier(max_depth=____, random_state=42)

# C. SVM dengan kernel RBF
svm = SVC(kernel="____", C=1.0)

# D. Mengukur F1 untuk kelas Churn=Yes
score = f1_score(y_test, y_pred, pos_label="____")

# E. Mengakses hyperparameter model di dalam pipeline
params = {"model__max_depth": [3, 5, 10]}
```

**Jawaban untuk cek mandiri:** A `7`, B `4`, C `rbf`, D `Yes`. Untuk E, pahami bahwa `model__max_depth` memakai dua underscore karena parameter `max_depth` milik step bernama `model`.